# 第4讲：高效 LLM 微调（二）——语料处理与可视化

第3讲中，我们通过 QLoRA 对 LLaMA-3 进行了领域微调，使模型能够准确解释数字人文专业术语。
然而，在真实项目中，视频转写语料并不是开箱即用的——它们充斥着**重复片段**、**格式噪声**和**未标注的术语**。

**本讲的目标**：在第3讲微调好的 LoRA 适配器基础上，构建一条从原始转写文本到可分析、可训练的高质量语料的完整数据流水线，并通过**精确率图表**直观呈现模型的术语识别效果。

# 本讲结构

## 理论部分（约 18 分钟）
### 第1章  环境准备与适配器加载
### 第2章  数字人文视频语料处理（去重与术语标注理论）
### 第3章  简单实验设计：固定超参数策略
### 第4章  精确率图表可视化理论

## 实践部分（约 12 分钟）
### 第5章  去重流水线实现
### 第6章  术语标注流水线实现
### 第7章  训练日志记录
### 第8章  精确率可视化实现
### 快速回顾

# 第1章  环境准备与适配器加载

本讲**复用第3讲保存的 LoRA 适配器**，无需从头微调模型。

加载流程：
1. 安装依赖（Colab 用户；本地用户自动跳过）
2. 配置路径并上传适配器文件
3. 加载基础模型 + LoRA 适配器并做推理检查

> **本地用户**：已在 `mllm-dh` 环境中安装好依赖，安装单元格会自动跳过。

In [ ]:
# ===== 第1章：安装依赖（仅 Colab 用户；本地自动跳过）=====
#
# 若安装后 import torch 出现 'RpcBackendOptions already defined'，
# 再执行一次 Runtime → Restart runtime 并跳过本单元格即可。

import subprocess
import sys

try:
    import google.colab  # noqa: F401
    _IN_COLAB_INSTALL = True
except ImportError:
    _IN_COLAB_INSTALL = False

if _IN_COLAB_INSTALL:
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers', 'accelerate', 'peft', 'bitsandbytes',
            'datasets', 'matplotlib', 'seaborn',
        ],
        check=True,
    )
    print('Colab 依赖安装完成，请继续运行下一个单元格。')
else:
    print('检测到本地环境：跳过 Colab 依赖安装。')

In [ ]:
# ===== 第1章：路径配置 =====
#
# 在 Google Colab 中，Python 内核的工作目录始终是 /content/，
# 无论 notebook 文件存放在哪个子文件夹里。
# 通过 files.upload() 上传的文件也会落在 /content/ 下。
# 本单元格自动检测运行环境并统一设置所有路径变量。

import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = '/content'
    os.chdir(BASE_DIR)
else:
    BASE_DIR = os.getcwd()

ADAPTER_PATH = os.path.join(BASE_DIR, 'qlora_adapter_lecture3')
LOGS_DIR     = os.path.join(BASE_DIR, 'logs')
os.makedirs(LOGS_DIR, exist_ok=True)

print(f'运行环境  : {"Google Colab" if IN_COLAB else "本地"}')
print(f'工作目录  : {os.getcwd()}')
print(f'适配器路径: {ADAPTER_PATH}')
print(f'日志目录  : {LOGS_DIR}')
print()
print('路径配置完成，继续运行下一个单元格。')

In [ ]:
# ===== 第1章：GPU 自检 =====

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '未检测到 GPU。\n'
        '  Colab 用户：Runtime → Change runtime type → T4 GPU\n'
        '  如果刚重启运行时，请确认已切换到 GPU 运行时类型。'
    )

device_name = torch.cuda.get_device_name(0)
total_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU 已就绪：{device_name}')
print(f'显存总量  ：{total_mem:.1f} GB')

In [ ]:
# ===== 第1章：上传适配器文件（仅 Colab 用户）=====
# 本讲不上传 zip，而是直接上传 shared/weights/qlora_adapter_lecture3_archive/
# 文件夹里的所有文件。Colab 会把这些文件保存到 /content/（即 BASE_DIR）。
# 本地用户可直接运行此单元格，它会自动跳过。

if IN_COLAB:
    from google.colab import files

    print('请选择 shared/weights/qlora_adapter_lecture3_archive/ 文件夹中的所有文件并上传。')
    print('至少应包含：adapter_config.json、adapter_model.safetensors，以及 tokenizer 相关文件。')

    uploaded = files.upload()  # 多选文件；文件会保存到 /content/

    if not uploaded:
        raise RuntimeError('未上传任何文件，请重新运行本单元格并选择适配器文件。')

    print()
    print(f'已上传 {len(uploaded)} 个文件到：{BASE_DIR}')
    print('文件列表：')
    for fname in sorted(uploaded.keys()):
        fpath = os.path.join(BASE_DIR, fname)
        fsize = os.path.getsize(fpath) / 1e6
        print(f'  {fname:<40} {fsize:>8.2f} MB')

    print()
    print('上传完成。下一个单元格会先尝试从 ADAPTER_PATH 加载；')
    print('如果该文件夹不存在，则自动从 BASE_DIR（/content/）加载这些单独上传的文件。')
else:
    print('检测到本地环境：跳过 Colab 文件上传。')

In [ ]:
# ===== 第1章：加载基础模型与 LoRA 适配器 =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'

# Determine the actual path to load the adapter from
load_path = ADAPTER_PATH
if not os.path.isdir(ADAPTER_PATH):
    # If the ADAPTER_PATH directory doesn't exist, assume files were uploaded directly to BASE_DIR
    print(f'Warning: Adapter directory not found at {ADAPTER_PATH}. Attempting to load from {BASE_DIR}.')
    load_path = BASE_DIR

required_adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
required_tokenizer_files = ['tokenizer_config.json']
optional_tokenizer_files = ['tokenizer.json', 'special_tokens_map.json']

missing_required = [
    fname for fname in required_adapter_files + required_tokenizer_files
    if not os.path.exists(os.path.join(load_path, fname))
]

if missing_required:
    available = sorted(os.listdir(load_path)) if os.path.isdir(load_path) else []
    raise FileNotFoundError(
        f'适配器文件不完整，加载路径：{load_path}\n'
        f'缺少必要文件：{missing_required}\n'
        '请确认已上传 shared/weights/qlora_adapter_lecture3_archive/ 中的所有文件。\n'
        f'当前路径下文件：{available}'
    )

missing_optional = [
    fname for fname in optional_tokenizer_files
    if not os.path.exists(os.path.join(load_path, fname))
]
if missing_optional:
    print(f'提示：未找到可选 tokenizer 文件：{missing_optional}')

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(load_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.config.use_cache = True

model = PeftModel.from_pretrained(base_model, load_path)
model.eval()

print('基础模型 + LoRA 适配器加载完成')
print(f'适配器路径：{load_path}')

In [ ]:
# ===== 检查点：验证适配器加载正确 =====

def run_inference(prompt: str, max_new_tokens: int = 120) -> str:
    """使用微调后的模型对单条输入做推理，返回生成文本。"""
    messages = [{'role': 'user', 'content': prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )

test_response = run_inference('视频中提到这部书是刻本，请解释这个术语。')
print('=== 适配器检查输出 ===')
print(test_response)
print('=' * 22)
print('适配器正常，可以继续。')

# 第2章  数字人文视频语料处理（理论）

## 2.1  为什么视频转写语料需要预处理？

学术讲座视频的自动转写结果通常包含以下噪声，若不处理将直接影响微调质量：

| 噪声类型 | 典型表现 | 对微调的危害 |
|----------|---------|-------------|
| **精确重复** | 同一段文字出现多次（幻灯片切换时 ASR 重复捕获） | 训练样本分布偏移，模型对高频文本过拟合 |
| **近似重复** | 措辞略有差异但语义相同的片段 | 浪费计算资源，降低数据多样性 |
| **术语未标注** | 「刻本」「金石学」混在普通句子中，无任何标记 | 模型无法感知哪些词是需要解释的专业术语 |
| **噪声文本** | 掌声、咳嗽、字幕乱码等 | 引入无效训练信号 |

## 2.2  去重策略

本讲实现两种去重方法，从简到繁：

**方法 A：精确去重（Exact Deduplication）**
- 将每条转写文本标准化（去除空白、统一标点）后，计算 SHA-256 哈希值
- 保留每个哈希的首次出现，丢弃后续重复
- 时间复杂度：O(n)，适合大规模语料

**方法 B：模糊去重（Fuzzy Deduplication）**
- 使用 MinHash + LSH（局部敏感哈希）估计句子级 Jaccard 相似度
- 相似度 ≥ 阈值（默认 0.8）的片段视为近似重复，保留最长版本
- 本讲用字符 n-gram 重叠率作为轻量替代，无需额外依赖

## 2.3  术语标注策略

标注的目标是在转写片段中识别并标记数字人文专业术语，为后续微调提供结构化的训练信号。

**字典匹配法（本讲采用）**：维护一份术语词典，对每条转写文本做多模式匹配（Aho-Corasick 算法的简化版本）。
- 优点：速度快、可解释、无需模型
- 缺点：召回率受词典覆盖率限制

**模型辅助标注（进阶，供参考）**：将微调后的 LLM 本身用于识别术语，构成「自标注」闭环。

# 第3章  简单实验设计：固定超参数策略（理论）

## 为什么固定超参数？

在探索性的数字人文实验中，**一次只改变一个变量**是获得可解释结论的基本原则。
对于课程阶段的微调实验，推荐固定以下超参数，将实验变量限定在**数据质量**（去重前 vs 去重后，无标注 vs 有标注）：

| 超参数 | 固定值 | 理由 |
|--------|--------|------|
| LoRA rank `r` | 16 | 数字人文词汇量适中，无需更高秩 |
| `lora_alpha` | 32 | 标准 2×r 设置 |
| 学习率 | 2e-4 | LoRA 微调经验最优区间 |
| batch size | 1 + grad_accum 4 | Colab T4 显存限制 |
| epoch | 3 | 小语料防止过拟合 |
| 随机种子 | 42 | 保证可复现性 |

## 实验对比矩阵

```
实验 A：原始语料（含重复）+ 无术语标注  ← 基线
实验 B：去重语料             + 无术语标注  ← 控制数据质量
实验 C：去重语料             + 有术语标注  ← 完整流水线
```

> 本讲用**仿真数据**演示三个实验的精确率差异，真实数据替换方式见第8章注释。

# 第4章  精确率图表可视化理论

## 4.1  评估指标：术语解释精确率

本讲使用**关键词命中率**作为轻量精确率指标：对于每条测试样本，检查模型的输出中是否包含标准答案中的核心关键词。

$$\text{精确率} = \frac{\text{关键词命中的样本数}}{\text{测试样本总数}}$$

这是一种**无参考模型的近似评估**，适合课程阶段的快速验证。生产环境中可替换为 ROUGE-L 或 BERTScore。

## 4.2  可视化选型

| 图表类型 | 用途 | 本讲是否使用 |
|----------|------|:-----------:|
| 折线图（epoch vs loss） | 训练动态监控 | 是（第7章）|
| 分组条形图（实验 vs 精确率） | 跨实验对比 | 是（第8章）|
| 热力图（术语 vs 模型） | 细粒度误差分析 | 是（第8章进阶）|

## 4.3  可视化设计原则（面向论文）

- 每张图必须有**标题、轴标签、单位**
- 误差棒（error bar）应标注标准差或 95% 置信区间
- 颜色方案选择对色觉障碍友好的调色板（如 `colorblind` 或 `tab10`）
- 图表输出 300 DPI，方便直接插入学术论文

# 第5章  去重流水线实现

## 实践目标

构建一个可对任意转写列表执行以下操作的 `CorpusCleaner` 类：
1. **精确去重**：基于哈希值
2. **模糊去重**：基于字符 n-gram 重叠率
3. **噪声过滤**：移除过短或过长的片段

In [ ]:
# ===== 第5章：语料去重工具类 =====
import hashlib
import re
from typing import List, Dict, Tuple


def normalize_text(text: str) -> str:
    """标准化文本：去除多余空白，统一中文标点。"""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('，', ',').replace('。', '.').replace('：', ':').replace('；', ';')
    return text


def text_hash(text: str) -> str:
    """计算标准化文本的 SHA-256 前 16 位哈希，用于精确去重。"""
    normalized = normalize_text(text)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()[:16]


def ngram_overlap(a: str, b: str, n: int = 3) -> float:
    """计算两段文本的字符 n-gram Jaccard 相似度，用于模糊去重。"""
    def get_ngrams(text: str, n: int):
        return set(text[i:i+n] for i in range(len(text) - n + 1))

    ngrams_a = get_ngrams(normalize_text(a), n)
    ngrams_b = get_ngrams(normalize_text(b), n)
    if not ngrams_a or not ngrams_b:
        return 0.0
    intersection = len(ngrams_a & ngrams_b)
    union = len(ngrams_a | ngrams_b)
    return intersection / union


class CorpusCleaner:
    """数字人文视频转写语料去重与过滤流水线。"""

    def __init__(self, min_chars: int = 10, max_chars: int = 400, fuzzy_threshold: float = 0.8):
        self.min_chars = min_chars
        self.max_chars = max_chars
        self.fuzzy_threshold = fuzzy_threshold

    def exact_dedup(self, texts: List[str]) -> Tuple[List[str], int]:
        seen, result = set(), []
        for text in texts:
            h = text_hash(text)
            if h not in seen:
                seen.add(h)
                result.append(text)
        return result, len(texts) - len(result)

    def fuzzy_dedup(self, texts: List[str]) -> Tuple[List[str], int]:
        texts_sorted = sorted(texts, key=len, reverse=True)
        kept = []
        for candidate in texts_sorted:
            if not any(ngram_overlap(candidate, ex) >= self.fuzzy_threshold for ex in kept):
                kept.append(candidate)
        return kept, len(texts) - len(kept)

    def length_filter(self, texts: List[str]) -> Tuple[List[str], int]:
        filtered = [t for t in texts if self.min_chars <= len(t) <= self.max_chars]
        return filtered, len(texts) - len(filtered)

    def clean(self, texts: List[str], verbose: bool = True) -> Tuple[List[str], Dict]:
        report = {'原始数量': len(texts)}
        texts, n = self.length_filter(texts);  report['长度过滤移除'] = n
        texts, n = self.exact_dedup(texts);    report['精确去重移除'] = n
        texts, n = self.fuzzy_dedup(texts);    report['模糊去重移除'] = n
        report['清洗后数量'] = len(texts)
        report['保留率'] = f"{100 * len(texts) / report['原始数量']:.1f}%"
        if verbose:
            print('=== 语料清洗报告 ===')
            for k, v in report.items():
                print(f'  {k}: {v}')
        return texts, report


print('CorpusCleaner 类定义完成')

In [ ]:
# ===== 第5章（续）：模拟真实语料并运行去重 =====

RAW_TRANSCRIPTS = [
    '视频中提到这部书是刻本，与手写本不同。',
    '文献里反复出现缮写一词，这是专职抄手对文本进行誊清的工序。',
    '研究者说他的研究方向是金石学，专注于铜器铭文与石刻碑文。',
    '档案整理时需要遵循原始顺序原则，保留文献背景关系。',
    '数字化之后，团队对每张图片做了详细的标注工作。',
    '不同版本之间存在异文，需要通过校勘恢复原始面貌。',
    '这是一部传世本，与出土文献有所不同，研究价值各有侧重。',
    '博物馆里展出了多件珍贵的拓片，这是传递金石文字最重要的媒介。',
    '他的博士论文研究的是目录学，关注文献的著录与分类方法。',
    '视频介绍了一处文化遗产的数字孪生项目，用三维扫描建立高精度模型。',
    # 精确重复（模拟 ASR 重复捕获）
    '视频中提到这部书是刻本，与手写本不同。',
    '档案整理时需要遵循原始顺序原则，保留文献背景关系。',
    # 近似重复（措辞略有差异）
    '视频中说这本书是刻本，和手写本是不一样的。',
    '老师提到档案整理需要遵循原始顺序原则，以保留内在背景关系。',
    # 噪声片段（过短）
    '嗯。', '好的好的。', '啊对对对。',
    # 正常长片段
    '甲骨文是殷商时期刻写在龟甲兽骨上的文字，是目前已知最早的成熟汉字系统，也是汉字演变研究的核心史料。',
    '活字印刷术由毕昇于北宋时期发明，将雕版整体雕刻改为单字排列组合，极大降低了书籍生产成本，是印刷史上的重要里程碑。',
    '敦煌壁画保存了大量唐代及更早期的宗教绘画与文字资料，是研究丝绸之路文明交流的第一手视觉文献。',
]

print(f'原始语料共 {len(RAW_TRANSCRIPTS)} 条')
print()
cleaner = CorpusCleaner(min_chars=10, max_chars=400, fuzzy_threshold=0.75)
cleaned_transcripts, clean_report = cleaner.clean(RAW_TRANSCRIPTS, verbose=True)
print()
print('清洗后示例（前3条）：')
for i, t in enumerate(cleaned_transcripts[:3]):
    print(f'  [{i+1}] {t}')

# 第6章  术语标注流水线实现

术语标注为每条转写片段生成两种输出：
1. **匹配到的术语列表**：用于统计术语分布
2. **带标记的文本**：在术语周围插入 `【】` 标记，作为微调数据的可选增强特征

In [ ]:
# ===== 第6章：术语词典与标注器 =====
from dataclasses import dataclass, field
from collections import Counter


DH_TERM_DICT = {
    '刻本': '版本学',    '缮写': '文献学',      '金石学': '传统学科',
    '原始顺序原则': '档案学', '校勘': '文献学',  '传世本': '版本学',
    '出土文献': '考古学', '拓片': '金石学',      '目录学': '传统学科',
    '数字孪生': '数字技术', '数字化标注': '数字人文', '异文': '文献学',
    '甲骨文': '古文字学', '活字印刷': '印刷史',  '敦煌壁画': '艺术史',
    '雕版': '印刷史',    '版本学': '传统学科',   '档案学': '传统学科',
    '碑帖': '金石学',    '铭文': '金石学',
}


@dataclass
class AnnotatedSample:
    original: str
    terms_found: List[str] = field(default_factory=list)
    categories: List[str] = field(default_factory=list)
    annotated_text: str = ''


class TermAnnotator:
    """基于字典匹配的术语标注器。"""

    def __init__(self, term_dict: Dict[str, str]):
        # 按术语长度降序排序，优先匹配更长的术语（避免「金石」遮蔽「金石学」）
        self.term_dict = dict(sorted(term_dict.items(), key=lambda x: len(x[0]), reverse=True))

    def annotate(self, text: str) -> AnnotatedSample:
        terms_found, categories, annotated = [], [], text
        for term, category in self.term_dict.items():
            if term in text and term not in terms_found:
                terms_found.append(term)
                categories.append(category)
                annotated = annotated.replace(term, f'【{term}】')
        return AnnotatedSample(original=text, terms_found=terms_found,
                               categories=categories, annotated_text=annotated)

    def annotate_corpus(self, texts: List[str]) -> Tuple[List[AnnotatedSample], Dict]:
        samples = [self.annotate(t) for t in texts]
        all_terms = [t for s in samples for t in s.terms_found]
        term_counter = Counter(all_terms)
        report = {
            '总片段数': len(samples),
            '含术语片段数': sum(1 for s in samples if s.terms_found),
            '术语总命中次数': len(all_terms),
            '不重复术语数': len(term_counter),
            '最高频术语 Top-5': term_counter.most_common(5),
        }
        return samples, report


print('TermAnnotator 类定义完成')

In [ ]:
# ===== 第6章（续）：运行术语标注并查看结果 =====

annotator = TermAnnotator(DH_TERM_DICT)
annotated_corpus, annotation_report = annotator.annotate_corpus(cleaned_transcripts)

print('=== 术语标注报告 ===')
for k, v in annotation_report.items():
    print(f'  {k}: {v}')

print()
print('=== 标注示例（前5条有术语的片段）===')
shown = 0
for sample in annotated_corpus:
    if sample.terms_found:
        print(f'  原文：{sample.original}')
        print(f'  标注：{sample.annotated_text}')
        print(f'  术语：{sample.terms_found}  类别：{sample.categories}')
        print()
        shown += 1
        if shown >= 5:
            break

# 第7章  训练日志记录

结构化的日志是实验可复现的基础。本章实现：
1. **实验元数据记录**：超参数、数据集信息、运行时间
2. **逐步损失记录**：从 Trainer 的 `log_history` 提取并保存为 JSON
3. **损失曲线绘制**：多实验对比的折线图

In [ ]:
# ===== 第7章：实验日志工具 =====
import json
import datetime
import subprocess
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm

# Colab 默认字体常缺少中文字形，会产生 font warning 或标题方框。
# 优先使用 Noto CJK；若 Colab 尚未安装，则在此处自动安装。
def configure_chinese_font():
    noto_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'

    if IN_COLAB and not os.path.exists(noto_path):
        print('正在安装 Noto CJK 字体以支持中文图表标题...')
        subprocess.run(['apt-get', '-qq', 'update'], check=False)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'], check=False)

    if os.path.exists(noto_path):
        fm.fontManager.addfont(noto_path)
        matplotlib.rcParams['font.family'] = ['Noto Sans CJK JP', 'DejaVu Sans']
        print('已启用 Noto CJK 字体。')
    else:
        matplotlib.rcParams['font.family'] = ['DejaVu Sans']
        print('未找到 Noto CJK 字体；图表仍可生成，但中文标题可能显示为方框。')

    matplotlib.rcParams['axes.unicode_minus'] = False


configure_chinese_font()


class ExperimentLogger:
    """轻量实验日志管理器，记录超参数与训练指标并支持可视化。"""

    def __init__(self, experiment_name: str):
        self.name = experiment_name
        os.makedirs(LOGS_DIR, exist_ok=True)
        self.record = {
            'experiment': experiment_name,
            'created_at': datetime.datetime.now().isoformat(),
            'hyperparameters': {},
            'dataset_info': {},
            'loss_history': [],
            'eval_metrics': {},
        }

    def log_hyperparameters(self, **kwargs): self.record['hyperparameters'].update(kwargs)
    def log_dataset_info(self, **kwargs):    self.record['dataset_info'].update(kwargs)
    def log_eval_metrics(self, **kwargs):    self.record['eval_metrics'].update(kwargs)

    def log_loss_history(self, trainer_log_history: List[Dict]):
        self.record['loss_history'] = [
            {'step': e['step'], 'loss': e['loss']}
            for e in trainer_log_history if 'loss' in e
        ]

    def save(self):
        path = os.path.join(LOGS_DIR, f'{self.name}.json')
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.record, f, ensure_ascii=False, indent=2)
        print(f'日志已保存：{path}')
        return path

    def print_summary(self):
        r = self.record
        print(f'=== 实验：{r["experiment"]} ===')
        print(f'  超参数：{r["hyperparameters"]}')
        print(f'  数据集：{r["dataset_info"]}')
        if r['loss_history']:
            print(f'  初始/最终损失：{r["loss_history"][0]["loss"]:.4f} → {r["loss_history"][-1]["loss"]:.4f}')
        if r['eval_metrics']:
            print(f'  评估指标：{r["eval_metrics"]}')


print('ExperimentLogger 类定义完成')

In [ ]:
# ===== 第7章（续）：仿真三组实验日志 =====

import numpy as np

np.random.seed(42)


def simulate_loss_curve(initial_loss, final_loss, n_steps, noise_scale=0.05):
    decay  = np.exp(np.linspace(0, -3, n_steps))
    losses = initial_loss + (final_loss - initial_loss) * (1 - decay)
    losses += np.random.normal(0, noise_scale, n_steps)
    losses  = np.clip(losses, 0.05, None)
    return [{'step': i + 1, 'loss': float(f'{l:.4f}')} for i, l in enumerate(losses)]


EXPERIMENT_CONFIGS = [
    {'name': 'exp_A_baseline',      'initial_loss': 2.8, 'final_loss': 1.15, 'accuracy': 0.52, 'dataset_size': 20, 'deduped': False, 'tagged': False},
    {'name': 'exp_B_deduped',       'initial_loss': 2.6, 'final_loss': 0.82, 'accuracy': 0.68, 'dataset_size': 15, 'deduped': True,  'tagged': False},
    {'name': 'exp_C_full_pipeline', 'initial_loss': 2.5, 'final_loss': 0.61, 'accuracy': 0.81, 'dataset_size': 15, 'deduped': True,  'tagged': True},
]

loggers = {}
for cfg in EXPERIMENT_CONFIGS:
    logger = ExperimentLogger(cfg['name'])
    logger.log_hyperparameters(lora_r=16, lora_alpha=32, learning_rate=2e-4,
                                batch_size=1, grad_accum=4, epochs=3, seed=42)
    logger.log_dataset_info(size=cfg['dataset_size'], deduped=cfg['deduped'], tagged=cfg['tagged'])
    logger.log_loss_history(simulate_loss_curve(cfg['initial_loss'], cfg['final_loss'], 30))
    logger.log_eval_metrics(term_accuracy=cfg['accuracy'])
    logger.save()
    loggers[cfg['name']] = logger
    logger.print_summary()
    print()

print('三组实验日志已创建。')

In [ ]:
# ===== 第7章（续）：损失曲线对比图 =====

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728', '#1f77b4', '#2ca02c']
labels_map = {
    'exp_A_baseline':      '实验 A：原始语料（基线）',
    'exp_B_deduped':       '实验 B：去重语料',
    'exp_C_full_pipeline': '实验 C：去重 + 术语标注',
}
for (name, logger), color in zip(loggers.items(), colors):
    h = logger.record['loss_history']
    ax.plot([x['step'] for x in h], [x['loss'] for x in h],
            color=color, linewidth=2.0, label=labels_map[name])

ax.set_xlabel('训练步骤', fontsize=12)
ax.set_ylabel('训练损失', fontsize=12)
ax.set_title('三组实验训练损失曲线对比\n（固定超参数，变量：数据处理策略）', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(left=1)
ax.set_ylim(bottom=0)

out = os.path.join(LOGS_DIR, 'loss_curves_comparison.png')
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'损失曲线已保存至 {out}')

# 第8章  精确率可视化实现

本章评估微调后模型在测试集上的**术语解释精确率**，并生成两种可视化图表：
1. **分组条形图**：跨实验精确率对比
2. **逐术语热力图**：定位模型在哪些术语上表现薄弱

In [ ]:
# ===== 第8章：构建测试集与精确率评估函数 =====

TEST_SET = [
    {'term': '刻本',         'input': '请解释视频转写中提到的「刻本」这一术语。',           'keywords': ['雕版', '印刷', '木版', '宋']},
    {'term': '缮写',         'input': '文献里出现了「缮写」这个词，请解释其含义。',         'keywords': ['抄手', '誊清', '抄录', '传播']},
    {'term': '金石学',       'input': '视频中研究者提到「金石学」，请解释这是什么学科。',   'keywords': ['铜器', '铭文', '石刻', '碑文', '考古']},
    {'term': '原始顺序原则', 'input': '档案整理中遵循的「原始顺序原则」是什么？',          'keywords': ['档案', '顺序', '背景', '证据']},
    {'term': '校勘',         'input': '研究者在对不同版本的文献做「校勘」，请解释。',      'keywords': ['比对', '异文', '版本', '文字']},
    {'term': '传世本',       'input': '什么是「传世本」，它与出土文献有何不同？',          'keywords': ['抄写', '刻印', '流传', '出土']},
    {'term': '拓片',         'input': '博物馆展出的「拓片」是怎样制作的？',               'keywords': ['宣纸', '墨', '石刻', '捶拓', '复制']},
    {'term': '目录学',       'input': '研究者的方向是「目录学」，请解释该学科的研究对象。', 'keywords': ['著录', '分类', '文献', '目录']},
    {'term': '数字孪生',     'input': '遗址保护项目中提到了「数字孪生」技术，请解释。',   'keywords': ['三维', '扫描', '数字模型', '虚拟', '文化遗产']},
    {'term': '甲骨文',       'input': '视频介绍了「甲骨文」，请解释其历史意义。',          'keywords': ['殷商', '龟甲', '汉字', '史料']},
]


def evaluate_response(response: str, keywords: List[str]) -> bool:
    return any(kw in response for kw in keywords)


def run_accuracy_eval(test_set: List[Dict], verbose: bool = True) -> Dict:
    results = []
    for item in test_set:
        response = run_inference(item['input'], max_new_tokens=150)
        hit = evaluate_response(response, item['keywords'])
        results.append({'term': item['term'], 'hit': hit, 'response': response})
        if verbose:
            print(f'  [{"命中" if hit else "未命中"}] {item["term"]}: {response[:60]}...')
    accuracy = sum(r['hit'] for r in results) / len(results)
    return {'accuracy': accuracy, 'per_term': results}


print(f'测试集构建完成，共 {len(TEST_SET)} 条')

In [ ]:
# ===== 第8章（续）：运行真实模型评估 =====
# 运行时间约 2–5 分钟（10 条 × 每条约 15–30 秒）

print('开始推理评估（共 10 条测试样本）...')
print()
real_eval = run_accuracy_eval(TEST_SET, verbose=True)
print()
print(f'=== 微调模型总体精确率：{real_eval["accuracy"]:.1%} ===')

print()
print('=== 结果解读 ===')
if real_eval['accuracy'] < 0.5:
    print('当前精确率偏低，这是小样本演示适配器的正常现象。')
    print('本讲重点是验证语料清洗、术语标注、日志记录与可视化流程是否跑通。')
    print('若要提升模型表现，请扩大第3讲训练集（建议 50+ 条）并保持评估集不参与训练。')
else:
    print('当前精确率已显示出一定领域适配效果。')
    print('仍建议检查逐术语热力图，定位哪些术语还需要补充训练样本。')

print('注意：实验 A/B 是仿真日志，用于展示实验设计趋势；实验 C 使用当前真实适配器评估结果。')

In [ ]:
# ===== 第8章（续）：用真实结果更新实验 C 日志 =====

loggers['exp_C_full_pipeline'].log_eval_metrics(
    term_accuracy=real_eval['accuracy'], note='真实模型评估结果',
)
loggers['exp_C_full_pipeline'].save()
print(f'实验 C 精确率（真实）：{real_eval["accuracy"]:.1%}')

In [ ]:
# ===== 第8章（续）：分组条形图——跨实验精确率对比 =====

experiment_labels = ['实验 A\n原始语料（基线）', '实验 B\n去重语料', '实验 C\n去重 + 术语标注']
accuracies = [
    loggers['exp_A_baseline'].record['eval_metrics']['term_accuracy'],
    loggers['exp_B_deduped'].record['eval_metrics']['term_accuracy'],
    loggers['exp_C_full_pipeline'].record['eval_metrics']['term_accuracy'],
]

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#d62728', '#1f77b4', '#2ca02c']
bars = ax.bar(experiment_labels, accuracies, color=bar_colors, width=0.5, edgecolor='white')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('术语解释精确率', fontsize=12)
ax.set_title('数据处理策略对微调效果的影响\n（LLaMA-3 + QLoRA，固定超参数）', fontsize=13)
ax.axhline(y=accuracies[0], color='#d62728', linestyle='--', alpha=0.4, linewidth=1)
ax.text(2.42, accuracies[0] + 0.01, '基线', color='#d62728', fontsize=9, alpha=0.7)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

out = os.path.join(LOGS_DIR, 'accuracy_bar_chart.png')
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'分组条形图已保存至 {out}')

In [ ]:
# ===== 第8章（续）：逐术语精确率热力图 =====
import seaborn as sns

per_term_results = real_eval['per_term']
terms      = [r['term'] for r in per_term_results]
hits       = [1 if r['hit'] else 0 for r in per_term_results]
hit_matrix = np.array(hits).reshape(1, -1)

fig, ax = plt.subplots(figsize=(12, 2.2))
cmap = matplotlib.colors.ListedColormap(['#d62728', '#2ca02c'])
sns.heatmap(
    hit_matrix, ax=ax, cmap=cmap, vmin=0, vmax=1,
    linewidths=2, linecolor='white',
    annot=[['命中' if h else '未命中' for h in hits]],
    fmt='', cbar=False, xticklabels=terms, yticklabels=['微调模型'],
)
ax.set_title(f'逐术语命中热力图（整体精确率 {real_eval["accuracy"]:.1%}）', fontsize=12, pad=10)
ax.set_xticklabels(terms, fontsize=10, rotation=20, ha='right')
ax.tick_params(left=False)

out = os.path.join(LOGS_DIR, 'per_term_heatmap.png')
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'逐术语热力图已保存至 {out}')

In [ ]:
# ===== 第8章（续）：综合仪表板——四图合一 =====

fig = plt.figure(figsize=(14, 10))
fig.suptitle('第4讲 · 实验综合报告\nLLaMA-3 QLoRA 数字人文术语微调', fontsize=14, y=0.98)

ax1 = fig.add_subplot(2, 2, 1)
colors_line = ['#d62728', '#1f77b4', '#2ca02c']
for (name, logger), col in zip(loggers.items(), colors_line):
    h = logger.record['loss_history']
    ax1.plot([x['step'] for x in h], [x['loss'] for x in h],
             color=col, linewidth=1.8, label=name.split('_', 1)[1])
ax1.set_xlabel('步骤', fontsize=10); ax1.set_ylabel('损失', fontsize=10)
ax1.set_title('训练损失曲线', fontsize=11); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(2, 2, 2)
b = ax2.bar(['A 基线', 'B 去重', 'C 完整'], accuracies, color=colors_line, width=0.5, edgecolor='white')
for bar, acc in zip(b, accuracies):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
             f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.0); ax2.set_ylabel('精确率', fontsize=10)
ax2.set_title('跨实验精确率对比', fontsize=11)
ax2.grid(axis='y', alpha=0.3); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

ax3 = fig.add_subplot(2, 1, 2)
cmap2 = matplotlib.colors.ListedColormap(['#d62728', '#2ca02c'])
sns.heatmap(
    hit_matrix, ax=ax3, cmap=cmap2, vmin=0, vmax=1,
    linewidths=2, linecolor='white',
    annot=[['命中' if h else '未命中' for h in hits]],
    fmt='', cbar=False, xticklabels=terms, yticklabels=['微调模型'],
)
ax3.set_title('逐术语命中情况', fontsize=11)
ax3.set_xticklabels(terms, fontsize=9, rotation=20, ha='right'); ax3.tick_params(left=False)

out = os.path.join(LOGS_DIR, 'dashboard.png')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'综合仪表板已保存至 {out}')

# 快速回顾（Quick Check）

**本讲完成了以下工作：**

- **语料去重**：实现精确去重（SHA-256 哈希）与模糊去重（字符 n-gram Jaccard 相似度）两种策略，构建 `CorpusCleaner` 类，支持任意视频转写语料的一键清洗

- **术语标注**：基于字典匹配构建 `TermAnnotator` 类，对清洗后的语料做多模式术语识别，生成带 `【】` 标记的增强文本和术语分布统计报告

- **实验设计**：固定 LoRA 超参数（r=16, lr=2e-4, seed=42），以数据处理策略（原始 / 去重 / 去重+标注）为唯一变量，设计三组可比实验

- **日志系统**：实现 `ExperimentLogger` 类，将超参数、数据集信息和逐步损失记录为结构化 JSON，保存在 `LOGS_DIR`（Colab: `/content/logs/`，本地: `./logs/`）

- **可视化输出**：生成三类学术级图表——训练损失折线图、跨实验精确率条形图、逐术语命中热力图，并整合为综合仪表板（`dashboard.png`）

> **与前三讲的连贯性**：L01 把转写文本变短变可读；L02 把画面与古籍对齐；L03 让模型认识领域术语；**L04 让数据本身变得更干净、更结构化，并建立了衡量模型进步的可视化标尺。**

---

**延伸练习**

1. 尝试将 `fuzzy_threshold` 从 0.75 调整到 0.6 或 0.9，观察清洗后语料数量的变化
2. 在 `DH_TERM_DICT` 中增加 5 个你熟悉的数字人文术语，重新运行标注流水线
3. 将 `evaluate_response` 中的关键词命中规则替换为 ROUGE-1 分数（提示：`from rouge_score import rouge_scorer`）